# LeetCode 1332 — Remove Palindromic Subsequences bằng Python

**Link đề:** https://leetcode.com/problems/remove-palindromic-subsequences/
**Độ khó:** Easy · **Chủ đề:** Two Pointers, String (bài "đánh lừa" — *trick question*)

```python
class Solution:
    def removePalindromeSub(self, s: str) -> int:
        ...
```

## Mục tiêu học
1. Phân biệt rõ **subsequence** (dãy con, không cần liên tiếp) và **substring** (chuỗi con liên tiếp) — mấu chốt của bài.
2. Ôn kiểm tra palindrome: slicing `s[::-1]` và **hai con trỏ** (two pointers).
3. Tự khám phá đáp án chỉ có thể là `1` hoặc `2` bằng **brute force BFS** trên chuỗi nhỏ, rồi **chứng minh**.
4. Trace hai con trỏ, Time/Space, lỗi hay gặp (hiểu nhầm thành substring).
5. Luyện biến thể: 3 loại ký tự, xoá substring palindrome, đếm bước bằng BFS.

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Cho chuỗi `s` chỉ gồm chữ `'a'` và `'b'`. Mỗi bước, ta được xoá **một dãy con (subsequence) palindrome** khỏi `s`.

Trả về **số bước ít nhất** để `s` thành rỗng.

- **Subsequence**: chuỗi tạo ra bằng cách xoá **một số** ký tự (có thể không xoá) **mà không đổi thứ tự** các ký tự còn lại. **Không cần liên tiếp.**
- **Palindrome**: đọc xuôi và ngược giống nhau.

| `s` | Output | Giải thích |
|---|---|---|
| `"ababa"` | `1` | cả chuỗi đã là palindrome → xoá một lần |
| `"abb"` | `2` | xoá `"a"` → `"bb"` → xoá `"bb"` |
| `"baabb"` | `2` | xoá `"baab"` → `"b"` → xoá `"b"` |

**Constraints:** `1 <= s.length <= 1000`, `s[i]` là `'a'` hoặc `'b'`.

Câu hỏi trước khi code (hint của đề):
- Chuỗi chỉ có 2 loại ký tự — điều này gợi ý gì?
- Một dãy con chỉ gồm **một loại** chữ cái (ví dụ `"aaaa"`) có luôn là palindrome không?

## 2. Nền tảng: subsequence vs substring

Với `s = "abab"`:
- `"aa"` là **subsequence** (lấy vị trí 0 và 2) nhưng **không** là substring (không liên tiếp).
- `"ba"` là cả substring (vị trí 1–2) lẫn subsequence.
- `"bb"` là subsequence (vị trí 1 và 3).

Ta viết hàm kiểm tra "`t` có phải subsequence của `s`?" bằng con trỏ: đi qua `s`, mỗi khi gặp ký tự trùng với ký tự tiếp theo cần tìm của `t` thì tiến con trỏ của `t`.

In [1]:
def is_subsequence(t: str, s: str) -> bool:
    j = 0                          # vị trí tiếp theo cần khớp trong t
    for ch in s:
        if j < len(t) and ch == t[j]:
            j += 1
    return j == len(t)

s = "abab"
for t in ["aa", "ba", "bb", "abab", "bba"]:
    print(f"{t!r:7} subsequence? {is_subsequence(t, s)!s:5} | substring? {t in s}")

'aa'    subsequence? True  | substring? False
'ba'    subsequence? True  | substring? True
'bb'    subsequence? True  | substring? False
'abab'  subsequence? True  | substring? True
'bba'   subsequence? False | substring? False


Ở trên, `t in s` với hai chuỗi kiểm tra **substring** (liên tiếp). Còn subsequence phải tự viết hàm.

### 2.1. Kiểm tra palindrome bằng slicing

`s[::-1]` là chuỗi đảo ngược (bước `-1`). `s == s[::-1]` ⇔ palindrome.

In [2]:
for s in ["ababa", "abb", "baab", "a", "ab"]:
    print(f"{s!r:8} đảo -> {s[::-1]!r:8} palindrome? {s == s[::-1]}")

'ababa'  đảo -> 'ababa'  palindrome? True
'abb'    đảo -> 'bba'    palindrome? False
'baab'   đảo -> 'baab'   palindrome? True
'a'      đảo -> 'a'      palindrome? True
'ab'     đảo -> 'ba'     palindrome? False


### 2.2. Kiểm tra palindrome bằng hai con trỏ

Đặt `left = 0`, `right = len(s) - 1`. Trong khi `left < right`: nếu `s[left] != s[right]` → không palindrome; ngược lại cùng tiến vào giữa. Không tạo chuỗi mới → bộ nhớ `O(1)`.

In [3]:
def is_palindrome_two_pointers(s: str) -> bool:
    left, right = 0, len(s) - 1
    while left < right:
        if s[left] != s[right]:
            return False
        left += 1
        right -= 1
    return True

for s in ["ababa", "abb", "baab", "a", "ab", "aa"]:
    print(f"{s!r:8} -> {is_palindrome_two_pointers(s)}")

'ababa'  -> True
'abb'    -> False
'baab'   -> True
'a'      -> True
'ab'     -> False
'aa'     -> True


### 2.3. Dãy con chỉ gồm một loại chữ luôn là palindrome

`"aaaa"` đọc ngược vẫn là `"aaaa"`. Và trong `s`, **tất cả** chữ `'a'` tạo thành một subsequence (lấy mọi vị trí có `'a'`, giữ thứ tự). Tương tự với `'b'`.

In [4]:
s = "baabbab"
all_a = ''.join(ch for ch in s if ch == 'a')
all_b = ''.join(ch for ch in s if ch == 'b')
print("mọi 'a':", all_a, "| subsequence?", is_subsequence(all_a, s), "| palindrome?", all_a == all_a[::-1])
print("mọi 'b':", all_b, "| subsequence?", is_subsequence(all_b, s), "| palindrome?", all_b == all_b[::-1])

mọi 'a': aaa | subsequence? True | palindrome? True
mọi 'b': bbbb | subsequence? True | palindrome? True


## 3. Cách trực giác / baseline — BFS thử mọi cách xoá (chỉ cho chuỗi nhỏ)

Nếu chưa nhận ra mẹo, ta có thể **mô phỏng** đúng định nghĩa: từ chuỗi hiện tại, thử xoá **mọi** subsequence palindrome không rỗng, xem cần ít bước nhất bao nhiêu để về rỗng.

Ta dùng **BFS** (tìm kiếm theo chiều rộng): đi theo "tầng" — tầng 0 là `s`, tầng 1 là mọi chuỗi còn lại sau 1 bước, ... Tầng đầu tiên chứa chuỗi rỗng chính là số bước tối thiểu.

Để liệt kê mọi subsequence, ta dùng **mặt nạ bit** (bitmask): với chuỗi dài `n`, mỗi số `mask` từ `1` đến `2^n - 1` chọn các vị trí `i` có bit `i` bằng 1. Rất tốn kém (`2^n`), chỉ dùng để **khám phá** với `n <= 8`.

In [5]:
s = "abb"
n = len(s)
for mask in range(1, 2 ** n):
    picked = ''.join(s[i] for i in range(n) if mask >> i & 1)   # bit i của mask bằng 1?
    print(f"mask={mask:03b} -> chọn {picked!r}")

mask=001 -> chọn 'a'
mask=010 -> chọn 'b'
mask=011 -> chọn 'ab'
mask=100 -> chọn 'b'
mask=101 -> chọn 'ab'
mask=110 -> chọn 'bb'
mask=111 -> chọn 'abb'


In [6]:
from collections import deque

def min_steps_bfs(s: str) -> int:
    start = s
    dist = {start: 0}
    queue = deque([start])
    while queue:
        cur = queue.popleft()
        if cur == "":
            return dist[cur]
        n = len(cur)
        for mask in range(1, 2 ** n):
            picked = ''.join(cur[i] for i in range(n) if mask >> i & 1)
            if picked != picked[::-1]:
                continue                                  # không phải palindrome
            rest = ''.join(cur[i] for i in range(n) if not (mask >> i & 1))
            if rest not in dist:
                dist[rest] = dist[cur] + 1
                queue.append(rest)
    return -1

for s in ["ababa", "abb", "baabb", "a", "ab", "abab"]:
    print(f"{s!r:8} -> BFS cho {min_steps_bfs(s)} bước")

'ababa'  -> BFS cho 1 bước
'abb'    -> BFS cho 2 bước
'baabb'  -> BFS cho 2 bước
'a'      -> BFS cho 1 bước
'ab'     -> BFS cho 2 bước
'abab'   -> BFS cho 2 bước


`deque` (hàng đợi hai đầu, module `collections`) cho `append` vào cuối và `popleft` lấy ra từ đầu đều `O(1)` — cấu trúc chuẩn cho BFS. `dict dist` vừa ghi số bước vừa đánh dấu chuỗi đã thăm.

### 3.1. Quan sát kết quả BFS trên **mọi** chuỗi a/b ngắn

In [7]:
from itertools import product

seen_answers = {}
for length in range(1, 8):
    for tup in product('ab', repeat=length):
        s = ''.join(tup)
        ans = min_steps_bfs(s)
        seen_answers.setdefault(ans, []).append(s)
for ans in sorted(seen_answers):
    print(f"đáp án {ans}: {len(seen_answers[ans])} chuỗi, ví dụ {seen_answers[ans][:5]}")

đáp án 1: 44 chuỗi, ví dụ ['a', 'b', 'aa', 'bb', 'aaa']
đáp án 2: 210 chuỗi, ví dụ ['ab', 'ba', 'aab', 'abb', 'baa']


**Quan sát:** đáp án chỉ là `1` hoặc `2`! Và đáp án là `1` đúng với những chuỗi... đã là palindrome. Ta kiểm tra giả thuyết đó:

In [8]:
ok = all((ans == 1) == (s == s[::-1]) for ans, lst in seen_answers.items() for s in lst)
print("đáp án 1  <=>  s là palindrome ?", ok)

đáp án 1  <=>  s là palindrome ? True


## 4. Nút thắt và insight

Nút thắt: BFS tốn `O(2^n)` — với `n = 1000` là không tưởng.

**Insight (đúng như hint):**
- Nếu `s` đã là palindrome → xoá cả chuỗi trong **1** bước.
- Ngược lại → bước 1 xoá **mọi chữ `'a'`** (subsequence `"aa...a"` là palindrome), bước 2 xoá **mọi chữ `'b'`**. Tổng **2** bước.
- Không thể ít hơn 1 (chuỗi không rỗng). Và nếu `s` không palindrome thì không thể xoá hết trong 1 bước (1 bước nghĩa là xoá cả `s`, đòi hỏi `s` là palindrome).

Vậy đáp án = `1` nếu `s` là palindrome, ngược lại `2`.

Bẫy lớn nhất: người đọc lướt tưởng là **substring** → nghĩ tới quy hoạch động phức tạp. Đọc kỹ chữ "subsequence"!

In [9]:
s = "baabbab"
print("s =", s, "| palindrome?", s == s[::-1])
step1 = ''.join(ch for ch in s if ch != 'a')
print("bước 1: xoá mọi 'a' ->", repr(step1))
step2 = ''.join(ch for ch in step1 if ch != 'b')
print("bước 2: xoá mọi 'b' ->", repr(step2))

s = baabbab | palindrome? False
bước 1: xoá mọi 'a' -> 'bbbb'
bước 2: xoá mọi 'b' -> ''


## 5. Lời giải chính — kiểm palindrome bằng hai con trỏ

**Đoạn 1:** `left = 0`, `right = len(s) - 1`.

**Đoạn 2:** trong khi `left < right`: nếu `s[left] != s[right]` → **trả `2` ngay**. Ngược lại tiến hai con trỏ.

**Đoạn 3:** đi hết mà không lệch → palindrome → trả `1`.

In [10]:
def remove_palindrome_sub(s: str) -> int:
    left, right = 0, len(s) - 1
    while left < right:
        if s[left] != s[right]:
            return 2
        left += 1
        right -= 1
    return 1

for s in ["ababa", "abb", "baabb", "a", "ab", "bbbb"]:
    print(f"{s!r:8} -> {remove_palindrome_sub(s)}")

'ababa'  -> 1
'abb'    -> 2
'baabb'  -> 2
'a'      -> 1
'ab'     -> 2
'bbbb'   -> 1


## 6. TRACE từng bước

Trace `s = "baabb"` (len 5):

| bước | left | right | s[left] | s[right] | so sánh | hành động |
|---|---|---|---|---|---|---|
| 1 | 0 | 4 | `b` | `b` | bằng | tiến vào |
| 2 | 1 | 3 | `a` | `b` | **khác** | trả `2` |

Trace `s = "ababa"`: cặp `(0,4)=a,a`, `(1,3)=b,b`, rồi `left = right = 2` → dừng, trả `1`.

In [11]:
def trace_remove(s: str) -> int:
    print("s =", repr(s))
    left, right = 0, len(s) - 1
    while left < right:
        same = s[left] == s[right]
        print(f"  left={left} ({s[left]}), right={right} ({s[right]}) -> {'bằng, tiến vào' if same else 'KHÁC -> trả 2'}")
        if not same:
            return 2
        left += 1
        right -= 1
    print(f"  left={left}, right={right}: hai con trỏ gặp/vượt nhau -> palindrome -> trả 1")
    return 1

trace_remove("baabb")
print()
trace_remove("ababa")
print()
trace_remove("abba")

s = 'baabb'
  left=0 (b), right=4 (b) -> bằng, tiến vào
  left=1 (a), right=3 (b) -> KHÁC -> trả 2

s = 'ababa'
  left=0 (a), right=4 (a) -> bằng, tiến vào
  left=1 (b), right=3 (b) -> bằng, tiến vào
  left=2, right=2: hai con trỏ gặp/vượt nhau -> palindrome -> trả 1

s = 'abba'
  left=0 (a), right=3 (a) -> bằng, tiến vào
  left=1 (b), right=2 (b) -> bằng, tiến vào
  left=2, right=1: hai con trỏ gặp/vượt nhau -> palindrome -> trả 1


1

## 7. Code nộp LeetCode

In [12]:
class Solution:
    def removePalindromeSub(self, s: str) -> int:
        left, right = 0, len(s) - 1
        while left < right:
            if s[left] != s[right]:
                return 2
            left += 1
            right -= 1
        return 1

sol = Solution()
print([sol.removePalindromeSub(s) for s in ["ababa", "abb", "baabb"]])

[1, 2, 2]


### Cách 2 — một dòng bằng slicing

```python
return 1 if s == s[::-1] else 2
```

Gọn nhất, nhưng tạo chuỗi đảo ngược `O(n)` bộ nhớ.

### Cách 3 — phiên bản tương thích đề cũ (cho phép chuỗi rỗng)

Phiên bản cũ của đề cho `0 <= s.length` — chuỗi rỗng cần `0` bước. Với constraints hiện tại (`>= 1`) không cần, nhưng viết thêm cho an toàn cũng không sai.

### Cách 4 — so sánh với `reversed` và `all`

`all(...)` trả `True` nếu mọi phần tử đều đúng. Ta chỉ cần so nửa đầu với nửa cuối.

In [13]:
class SolutionSlice:
    def removePalindromeSub(self, s: str) -> int:
        return 1 if s == s[::-1] else 2

class SolutionLegacy:
    def removePalindromeSub(self, s: str) -> int:
        if not s:                 # chuỗi rỗng (đề cũ)
            return 0
        return 1 if s == s[::-1] else 2

class SolutionAll:
    def removePalindromeSub(self, s: str) -> int:
        n = len(s)
        return 1 if all(s[i] == s[n - 1 - i] for i in range(n // 2)) else 2

for cls in [SolutionSlice, SolutionLegacy, SolutionAll]:
    print(cls.__name__, [cls().removePalindromeSub(s) for s in ["ababa", "abb", "baabb"]])
print("đề cũ, chuỗi rỗng:", SolutionLegacy().removePalindromeSub(""))

SolutionSlice [1, 2, 2]
SolutionLegacy [1, 2, 2]
SolutionAll [1, 2, 2]
đề cũ, chuỗi rỗng: 0


### Bảng so sánh

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| BFS (khám phá) | thử mọi cách xoá | `O(2^n · ...)` | rất lớn | chỉ để quan sát, `n <= 8` |
| 1. Hai con trỏ | dừng sớm khi lệch | `O(n)` | `O(1)` | lời giải chính |
| 2. Slicing | `s == s[::-1]` | `O(n)` | `O(n)` | ngắn nhất |
| 3. Legacy | thêm `if not s: 0` | `O(n)` | `O(n)` | an toàn cho đề cũ |
| 4. `all` nửa chuỗi | generator, dừng sớm | `O(n)` | `O(1)` | phong cách Python |

## 8. Vì sao đúng? (chứng minh)

Gọi `ans` là số bước tối thiểu, `s` không rỗng.

1. **`ans >= 1`**: chuỗi không rỗng thì cần ít nhất một bước.
2. **`ans == 1` ⇔ `s` là palindrome**: một bước xoá hết nghĩa là subsequence bị xoá chính là toàn bộ `s` → `s` phải là palindrome. Ngược lại nếu `s` palindrome thì xoá luôn cả `s`.
3. **`ans <= 2` luôn đúng**: xoá subsequence gồm mọi `'a'` (palindrome vì chỉ một loại chữ), rồi xoá mọi `'b'`. Nếu một trong hai loại không xuất hiện thì `s` toàn một chữ → đã là palindrome → rơi vào trường hợp 2.

Kết hợp: `ans = 1` nếu `s` palindrome, ngược lại `ans = 2`. Kết quả BFS ở mục 3.1 xác nhận trên mọi chuỗi độ dài ≤ 7. ∎

**Invariant của vòng hai con trỏ:** trước mỗi vòng, `s[:left]` là đảo ngược của `s[right+1:]` (các cặp đã so đều khớp).

### 8.1. Độ phức tạp

- **Time:** `O(n)` — tối đa `n/2` lần so sánh, dừng sớm khi lệch.
- **Space:** `O(1)` với hai con trỏ; `O(n)` với slicing.

## 9. Bộ test

Mẫu đề, biên (1 ký tự), toàn một chữ, độ dài chẵn/lẻ, chuỗi dài 1000, và **đối chiếu** lời giải với BFS trên **mọi** chuỗi a/b độ dài 1..7.

In [14]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<20} PASS -> {actual}")

check("ví dụ 1", sol.removePalindromeSub("ababa"), 1)
check("ví dụ 2", sol.removePalindromeSub("abb"), 2)
check("ví dụ 3", sol.removePalindromeSub("baabb"), 2)
check("1 ký tự", sol.removePalindromeSub("b"), 1)
check("toàn 'a'", sol.removePalindromeSub("aaaa"), 1)
check("chẵn, palindrome", sol.removePalindromeSub("abba"), 1)
check("chẵn, không", sol.removePalindromeSub("ab"), 2)
check("dài 1000 palin", sol.removePalindromeSub("ab" * 250 + "ba" * 250), 1)
check("dài 1000 không", sol.removePalindromeSub("a" * 999 + "b"), 2)

ví dụ 1              PASS -> 1
ví dụ 2              PASS -> 2
ví dụ 3              PASS -> 2
1 ký tự              PASS -> 1
toàn 'a'             PASS -> 1
chẵn, palindrome     PASS -> 1
chẵn, không          PASS -> 2
dài 1000 palin       PASS -> 1
dài 1000 không       PASS -> 2


In [15]:
methods = [remove_palindrome_sub, sol.removePalindromeSub, SolutionSlice().removePalindromeSub,
           SolutionLegacy().removePalindromeSub, SolutionAll().removePalindromeSub]
count = 0
for ans, lst in seen_answers.items():
    for s in lst:
        for f in methods:
            assert f(s) == ans, (f, s)
        count += 1
print("Khớp BFS trên", count, "chuỗi (mọi chuỗi a/b dài 1..7)")

Khớp BFS trên 254 chuỗi (mọi chuỗi a/b dài 1..7)


## 10. Lỗi người mới hay gặp

### Lỗi 1: Hiểu nhầm thành **substring** → nghĩ đáp án có thể lớn

Nếu chỉ được xoá **substring** (liên tiếp) palindrome, đáp án có thể vượt 2. Ta viết BFS cho phiên bản substring để so sánh (ví dụ `"aababb"`, `"aabbab"` cần 3 bước):

In [16]:
def min_steps_bfs_substring(s: str) -> int:
    dist = {s: 0}
    queue = deque([s])
    while queue:
        cur = queue.popleft()
        if cur == "":
            return dist[cur]
        for i in range(len(cur)):
            for j in range(i + 1, len(cur) + 1):
                sub = cur[i:j]
                if sub == sub[::-1]:
                    rest = cur[:i] + cur[j:]
                    if rest not in dist:
                        dist[rest] = dist[cur] + 1
                        queue.append(rest)
    return -1

for s in ["abb", "abab", "aababb", "aabbab"]:
    print(f"{s!r:9} subsequence: {remove_palindrome_sub(s)} | substring: {min_steps_bfs_substring(s)}")

'abb'     subsequence: 2 | substring: 2
'abab'    subsequence: 2 | substring: 2
'aababb'  subsequence: 2 | substring: 3
'aabbab'  subsequence: 2 | substring: 3


Bản substring là một bài **khác hẳn** (khó hơn nhiều, gần với LeetCode 1246 Palindrome Removal — Hard). Đọc kỹ đề!

### Lỗi 2: Đếm số "đoạn" ký tự liền nhau — vì nghĩ phải xoá từng khúc liên tiếp

In [17]:
def wrong_count_runs(s):
    # đếm số "đoạn" ký tự giống nhau liên tiếp — tư duy sai
    runs = 1
    for i in range(1, len(s)):
        if s[i] != s[i - 1]:
            runs += 1
    return runs

for s in ["ababa", "abb", "baabb"]:
    print(f"{s!r:8} sai: {wrong_count_runs(s)} | đúng: {remove_palindrome_sub(s)}")

'ababa'  sai: 5 | đúng: 1
'abb'    sai: 2 | đúng: 2
'baabb'  sai: 3 | đúng: 2


### Lỗi 3: Quên tiến con trỏ trong vòng hai con trỏ

Viết `while left < right:` mà quên `left += 1; right -= 1` khi hai ký tự bằng nhau → **vòng lặp vô hạn**. Luôn tự hỏi: "mỗi vòng, khoảng `[left, right]` có hẹp lại không?"

### Lỗi 4: Nghĩ rằng toàn một loại chữ cần 1 bước nhưng có 2 loại thì luôn 2

`"aba"` có 2 loại chữ nhưng là palindrome → chỉ **1** bước. Điều kiện đúng là "palindrome hay không", không phải "số loại chữ".

In [18]:
def wrong_by_kinds(s):
    return len(set(s))

for s in ["aba", "abb", "aaaa"]:
    print(f"{s!r:7} sai: {wrong_by_kinds(s)} | đúng: {remove_palindrome_sub(s)}")

'aba'   sai: 2 | đúng: 1
'abb'   sai: 2 | đúng: 2
'aaaa'  sai: 1 | đúng: 1


## 11. Mẹo nhận dạng pattern

- Đề Easy mà mô tả nghe "khó" (xoá subsequence tối ưu) ⇒ nghi ngờ có **mẹo** — đáp án nhỏ cố định.
- **Bảng chữ cái rất nhỏ** (chỉ `a`, `b`) là tín hiệu mạnh: thử nghĩ "gom hết một loại chữ".
- **Subsequence** ≠ **substring**: gạch chân từ khoá khi đọc đề.
- Không chắc chắn? Viết **brute force nhỏ** (BFS) để in đáp án cho mọi input ngắn, tìm quy luật, rồi chứng minh.
- Kiểm palindrome: `s == s[::-1]` (nhanh viết) hoặc hai con trỏ (bộ nhớ `O(1)`).

## 12. Biến thể

1. **Ba loại chữ `a`, `b`, `c`**: đáp án ≤ 3 (xoá từng loại), = 1 nếu palindrome; nhưng có thể = 2 — khó hơn (cần kiểm tra thêm).
2. **Xoá substring palindrome** — bài khác hẳn, cần quy hoạch động.
3. **LeetCode 125 Valid Palindrome / 680 Valid Palindrome II**: luyện hai con trỏ.

In [19]:
def valid_palindrome_ii(s: str) -> bool:
    # LeetCode 680: được xoá tối đa 1 ký tự
    def is_pal(l, r):
        while l < r:
            if s[l] != s[r]:
                return False
            l += 1; r -= 1
        return True
    l, r = 0, len(s) - 1
    while l < r:
        if s[l] != s[r]:
            return is_pal(l + 1, r) or is_pal(l, r - 1)
        l += 1; r -= 1
    return True

for s in ["aba", "abca", "abc"]:
    print(s, valid_palindrome_ii(s))

aba True
abca True
abc False


## 13. Bài tập tự luyện (TODO — tăng dần)

**Bài 1 (tính tay):** Đáp án cho `"aabbaa"`, `"aabbab"`, `"b"`, `"ba"`?

**Bài 2 (đoán output):** `is_subsequence("ab", "ba")` và `"ab" in "aab"`?

**Bài 3 (tự viết):** `exercise_remove(s)` bằng hai con trỏ.

**Bài 4 (tự viết):** `exercise_two_steps(s)` — trả về **danh sách 2 subsequence** cần xoá (dạng chuỗi) khi `s` không palindrome, hoặc `[s]` khi palindrome.

**Bài 5 (biến thể):** `exercise_remove_abc(s)` — `s` gồm `a`, `b`, `c`; trả về một **cận trên** đơn giản: 1 nếu palindrome, ngược lại số loại chữ xuất hiện.

**Bài 6 (mini challenge):** `exercise_is_subsequence(t, s)` — tự viết lại hàm kiểm tra subsequence bằng hai con trỏ `i`, `j`.

In [20]:
def exercise_remove(s: str) -> int:
    # TODO
    pass

def exercise_two_steps(s: str) -> list[str]:
    # TODO
    pass

# assert exercise_remove("ababa") == 1 and exercise_remove("abb") == 2
# assert exercise_two_steps("abb") == ["a", "bb"]
# assert exercise_two_steps("aba") == ["aba"]

In [21]:
def exercise_remove_abc(s: str) -> int:
    # TODO
    pass

def exercise_is_subsequence(t: str, s: str) -> bool:
    # TODO
    pass

# assert exercise_remove_abc("abc") == 3 and exercise_remove_abc("aca") == 1 and exercise_remove_abc("aab") == 2
# assert exercise_is_subsequence("aa", "abab") and not exercise_is_subsequence("bba", "abab")

## 14. Gợi ý 3 mức

**Bài 1** — (1) Chuỗi nào là palindrome? (2) Đọc ngược từng chuỗi. (3) Palindrome → 1, còn lại → 2.

**Bài 2** — (1) Subsequence giữ thứ tự. (2) `"ba"` có `'a'` đứng sau `'b'`... (3) `in` giữa hai chuỗi là substring.

**Bài 4** — (1) Kiểm palindrome trước. (2) Gom các `'a'` và các `'b'`. (3) `''.join(ch for ch in s if ch == 'a')`.

**Bài 5** — (1) Palindrome → 1. (2) Mỗi loại chữ là một subsequence palindrome. (3) `len(set(s))`.

**Bài 6** — (1) Con trỏ `i` trên `t`, `j` trên `s`. (2) Khớp thì tăng `i`; luôn tăng `j`. (3) Kết thúc: `i == len(t)`.

## 15. Đáp án tham khảo

**Bài 1:** `"aabbaa"` → 1; `"aabbab"` → 2; `"b"` → 1; `"ba"` → 2.

**Bài 2:** `False` — trong `"ba"`, chữ `'a'` duy nhất đứng ở cuối nên không còn `'b'` nào sau nó; và `True` — `"ab"` xuất hiện liên tiếp ở vị trí 1–2 của `"aab"`.

In [22]:
def answer_remove(s):
    l, r = 0, len(s) - 1
    while l < r:
        if s[l] != s[r]:
            return 2
        l += 1; r -= 1
    return 1

def answer_two_steps(s):
    if s == s[::-1]:
        return [s]
    return [''.join(ch for ch in s if ch == 'a'), ''.join(ch for ch in s if ch == 'b')]

def answer_remove_abc(s):
    return 1 if s == s[::-1] else len(set(s))

def answer_is_subsequence(t, s):
    i = j = 0
    while i < len(t) and j < len(s):
        if t[i] == s[j]:
            i += 1
        j += 1
    return i == len(t)

assert [answer_remove(x) for x in ["aabbaa", "aabbab", "b", "ba"]] == [1, 2, 1, 2]
assert is_subsequence("ab", "ba") is False and ("ab" in "aab") is True
assert answer_remove("ababa") == 1 and answer_remove("abb") == 2
assert answer_two_steps("abb") == ["a", "bb"] and answer_two_steps("aba") == ["aba"]
assert answer_remove_abc("abc") == 3 and answer_remove_abc("aca") == 1 and answer_remove_abc("aab") == 2
assert answer_is_subsequence("aa", "abab") and not answer_is_subsequence("bba", "abab")
# kiểm tra: 2 bước của answer_two_steps thực sự xoá hết và mỗi phần là palindrome
for lst in seen_answers.values():
    for s in lst:
        parts = answer_two_steps(s)
        assert len(parts) == answer_remove(s) and all(p == p[::-1] for p in parts)
        assert sorted(''.join(parts)) == sorted(s)
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 16. Template nhớ nhanh + checklist submit

```python
class Solution:
    def removePalindromeSub(self, s: str) -> int:
        return 1 if s == s[::-1] else 2      # palindrome: 1 bước; còn lại: xoá hết 'a' rồi hết 'b'
```

**Checklist:**
- [ ] Đọc đúng chữ **subsequence** (không cần liên tiếp).
- [ ] Hiểu vì sao đáp án ≤ 2 (mỗi loại chữ là một palindrome).
- [ ] Hai con trỏ: `left < right`, nhớ tiến cả hai.
- [ ] Đã test `"ababa"`, `"abb"`, 1 ký tự, toàn một chữ.